# Laboratory 16 — Thermal radiation and the heat capacity of solids

In this laboratory you rebuild Planck's law from the pieces the module derives — module 12's
oscillator, and a count of the standing waves in a box — and then test it. You count the modes
of a cavity one by one; find where the spectrum peaks, per unit frequency and per unit
wavelength, and discover that the two answers are different photons; integrate the spectrum
numerically and watch the $T^4$ law and its constant appear; and then turn to solids, where
the same formula, summed over a crystal's vibrations, is fitted to the measured heat capacity of
silver from $1\ \mathrm{K}$ to $300\ \mathrm{K}$. You finish by fitting the cosmic microwave
background, measured by NASA's COBE satellite, and by quoting two measurements with their
uncertainties stated honestly.

The route: the objects (Part 1); one mode, rebuilt from module 12 (Part 2); counting the modes of
a cavity (Part 3); Planck's law as a product (Part 4); two spectra, two peaks (Part 5); the
Stefan–Boltzmann law by quadrature (Part 6); silver at low temperature (Part 7); Einstein against
Debye (Part 8); the cosmic microwave background (Part 9); the automated checks (Part 10); and two
explorers (Part 11).

Work through it in order. Where the notebook asks you to predict, write your prediction in the
cell provided **before** running the next cell. A prediction you have committed to is the only
reliable way to discover that you were wrong.

**Units.** SI throughout: frequencies in hertz, wavelengths in metres, spectra in
$\mathrm{J\,s\,m^{-3}}$ (per unit frequency) or $\mathrm{J\,m^{-4}}$ (per unit wavelength), heat
capacities in $\mathrm{J\,K^{-1}}$ — per mole when the number of atoms is $N_A$.

## Model specification

| | |
|---|---|
| **System** | the electromagnetic field in a cavity of volume $V$ at temperature $T$, as independent standing-wave modes with energies in whole multiples of $h\nu$; a crystal of $N$ atoms as $3N$ such oscillators, at one frequency (Einstein) or spread over a cut-off sound spectrum (Debye) |
| **Dynamics** | none — every curve is thermal equilibrium |
| **Boundary** | perfectly reflecting walls held at $T$; a small hole emits a flux $c/4$ times the energy density |
| **Ensemble** | canonical, mode by mode; photon number not conserved, so $\mu = 0$ |
| **Ignored** | zero-point energy, photon–photon interaction, emissivity below one; for solids, real phonon dispersion, thermal expansion ($C_P - C_V$) and the conduction electrons (until Part 7 measures them) |
| **Valid when** | the cavity is much larger than $hc/(\kB T)$; the crystal is harmonic and warm enough that the electrons' $\gamma T$ is negligible |
| **Failure modes** | tiny or ultracold cavities; non-thermal light; metals below a few kelvin; crystals near melting |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thermolab import partition, radiation
from thermolab.constants import K_B, N_A, R_GAS
from thermolab.validation import relative_error, scaling_exponent

H, C = radiation.PLANCK_H, radiation.C_LIGHT

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

print(f"Wien roots: x = {radiation.WIEN_X_NU:.6f} (per frequency), "
      f"{radiation.WIEN_X_LAMBDA:.6f} (per wavelength)")
print(f"a = {radiation.RADIATION_CONSTANT:.4e} J m^-3 K^-4,  "
      f"sigma = {radiation.STEFAN_BOLTZMANN:.4e} W m^-2 K^-4")

## Part 1 — The objects

| function | what it gives |
|---|---|
| `mode_occupation(quantum, T)`, `mean_mode_energy(nu, T)` | $\langle n \rangle = 1/(e^{x} - 1)$ and $h\nu\langle n \rangle$ for one mode, $x = h\nu/(\kB T)$ |
| `mode_density(nu, V)`, `count_cavity_modes(L, nu_max)`, `mode_count_smooth(L, nu_max)` | $g(\nu) = 8\pi V\nu^2/c^3$; the modes of a cube counted one by one; the smooth count $8\pi V\nu^3/(3c^3)$ |
| `planck_u_nu(nu, T)`, `planck_u_lambda(lam, T)`, `rayleigh_jeans_u_nu(nu, T)` | the two spectral densities, and the classical one |
| `wien_peak_nu(T)`, `wien_peak_lambda(T)` | where each spectrum peaks |
| `integrate_u_nu(T, n)`, `integrate_u_lambda(T, n)`, `energy_density(T)`, `simpson(y, x)` | the total energy by quadrature in either variable, and $aT^4$ |
| `photon_gas(T)`, `photon_number_density(T)` | $u$, $P$, $s$ and $n$ of the photon gas |
| `einstein_heat_capacity(T, theta_E, N)`, `debye_heat_capacity(T, theta_D, N)`, `debye_low_t(...)` | the two models of a solid, and Debye's $T^3$ law |
| `fit_einstein_temperature`, `fit_debye_temperature`, `fit_blackbody_temperature` | one-parameter least-squares fits; each returns `.value`, `.sigma`, `.residuals`, `.rms_residual`, `.chi_squared` |

### Predict

Commit to an answer for each before running anything.

1. A mode with $h\nu = 3\kB T$: does it hold about $\kB T$, about $0.15\,\kB T$, or nothing?
2. At $5772\ \mathrm{K}$ the spectrum per unit wavelength peaks at $502\ \mathrm{nm}$. Per unit
   frequency, does it peak at the frequency of $502\ \mathrm{nm}$ light?
3. Heating from $300$ to $600\ \mathrm{K}$: by what factor does the total energy grow?
4. Silver's heat capacity between $1$ and $5\ \mathrm{K}$: on a log–log plot, what slope?

**Your predictions:**

1.
2.
3.
4.

## Part 2 — One mode, rebuilt from module 12

Module 12 summed the Boltzmann factors of an oscillator's levels. Here that explicit sum, run
over $4000$ levels $n\,h\nu$, is set against the closed form $\langle n \rangle = 1/(e^x - 1)$,
and the mean energy is plotted against the classical $\kB T$.

**Your prediction:** at what value of $h\nu/(\kB T)$ does the mean energy fall to half of $\kB T$?

*(write here before running the next cell)*

In [ ]:
quantum = 4e-21  # J
for temperature in (30.0, 300.0, 3000.0):
    levels = quantum * np.arange(4000)
    brute = partition.level_sums(levels, temperature).energy / quantum
    closed = radiation.mode_occupation(quantum, temperature)
    print(f"T = {temperature:6.0f} K: explicit sum <n> = {brute:.12f}, "
          f"closed form {closed:.12f}, relative difference {relative_error(closed, brute):.1e}")

x = np.linspace(0.01, 8, 400)
t_ref = 300.0
energy = radiation.mean_mode_energy(x * K_B * t_ref / H, t_ref) / (K_B * t_ref)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, energy, color="#2563eb", lw=2, label="quantum: x / (e^x - 1)")
ax.axhline(1.0, color="#dc2626", ls="--", label="classical: k_B T")
ax.set_xlabel("h nu / (k_B T)")
ax.set_ylabel("<E> / (k_B T)")
ax.legend()
plt.tight_layout()
plt.show()
half = x[np.argmin(np.abs(energy - 0.5))]
print(f"<E> falls to k_B T / 2 at h nu = {half:.2f} k_B T; "
      f"at 3 k_B T it is {3 / np.expm1(3):.3f} k_B T")

The closed form is module 12's sum, to twelve digits. A cheap quantum ($x \ll 1$) holds the
classical $\kB T$; a dear one holds exponentially little. That is the whole quantum input.

## Part 3 — Counting the modes of a cavity

A standing wave in a cube of side $L$ has $\mathbf{k} = (\pi/L)(n_x, n_y, n_z)$ with positive
integers, and two polarizations. `count_cavity_modes` enumerates the integer triples one by one;
the derivation replaced them by the volume of an octant. How good is that, and how does it
improve?

In [ ]:
side = 1.0
print(" radius |n|   explicit count     smooth count   shortfall")
shortfalls = {}
for radius in (5, 10, 20, 40, 80, 160, 320):
    nu = radius * C / (2 * side)
    counted = radiation.count_cavity_modes(side, nu)
    smooth = radiation.mode_count_smooth(side, nu)
    shortfalls[radius] = 1 - counted / smooth
    print(f"{radius:10d} {counted:16d} {smooth:16.0f}   {100 * shortfalls[radius]:6.2f} %")
print(f"shortfall x radius: {[round(r * s, 2) for r, s in shortfalls.items()]}")
print(f"a 1 cm cavity at 300 K: |n| near the peak = "
      f"{2 * 0.01 * radiation.wien_peak_nu(300.0) / C:.0f}")

The shortfall times the radius is nearly constant: the error is a surface effect, falling as
$1/|\mathbf{n}|$. For any cavity you could build, at any temperature you could reach, the
smooth count is exact for practical purposes.

## Part 4 — Planck's law as a product

Multiply the number of modes per unit volume per unit frequency, $g(\nu)/V$, by the mean energy
per mode. Do it once with the quantum mean energy and once with the classical $\kB T$, and compare
with `planck_u_nu`.

**Your prediction:** where do the classical and quantum curves agree, and where do they part?

*(write here before running the next cell)*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colours = {1500.0: "#d97706", 3000.0: "#dc2626", 6000.0: "#2563eb"}
for temperature, colour in colours.items():
    nu = np.linspace(0, 3.5 * radiation.wien_peak_nu(6000.0), 800)
    product = radiation.mode_density(nu) * radiation.mean_mode_energy(nu, temperature)
    assert np.allclose(product, radiation.planck_u_nu(nu, temperature), rtol=1e-14, atol=0)
    ax.plot(nu / 1e12, product, color=colour, lw=2, label=f"Planck, {temperature:.0f} K")
    ax.plot(nu / 1e12, radiation.rayleigh_jeans_u_nu(nu, temperature), ls="--", color=colour,
            lw=1, label=f"classical, {temperature:.0f} K")
ax.axvspan(C / 750e-9 / 1e12, C / 380e-9 / 1e12, color="#fde68a", alpha=0.4, label="visible")
ax.set_ylim(0, 1.3 * radiation.planck_u_nu(radiation.wien_peak_nu(6000.0), 6000.0))
ax.set_xlabel("frequency nu (THz)")
ax.set_ylabel("u_nu (J s m^-3)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

for temperature in (500.0, 3000.0):
    nu_v = np.array([C / 750e-9, C / 380e-9])
    classical = 8 * np.pi * K_B * temperature / (3 * C**3) * (nu_v[1]**3 - nu_v[0]**3)
    total = radiation.energy_density(temperature)
    quantum = radiation.band_fraction(temperature, 380e-9, 750e-9) * total
    print(f"{temperature:5.0f} K, visible band: classical {classical:.3e} J/m^3, "
          f"Planck {quantum:.3e} J/m^3, whole Planck spectrum {total:.3e} J/m^3")

The product *is* Planck's law, to rounding. The classical curves hug the quantum ones at low
frequency and then climb without limit; at $500\ \mathrm{K}$ the classical theory puts thousands
of times more energy into the visible band alone than the whole real spectrum holds.

## Part 5 — Two spectra, two peaks

Locate the peak of $u_\nu$ and of $u_\lambda$ by brute force — the largest value on a fine grid —
at several temperatures, and compare $\nu_{\max}\lambda_{\max}$ with $c$.

**Your prediction:** what is $\nu_{\max}\lambda_{\max}/c$?

*(write here before running the next cell)*

In [ ]:
products = []
print("     T (K)   lambda_max (nm)   nu_max (THz)   c / nu_max (nm)   nu_max lambda_max / c")
for temperature in (300.0, 1000.0, 3000.0, 5772.0, 10000.0):
    nu = np.linspace(0.2, 3.0, 200_001) * radiation.wien_peak_nu(temperature)
    lam = np.linspace(0.2, 3.0, 200_001) * radiation.wien_peak_lambda(temperature)
    nu_peak = nu[np.argmax(radiation.planck_u_nu(nu, temperature))]
    lam_peak = lam[np.argmax(radiation.planck_u_lambda(lam, temperature))]
    products.append(nu_peak * lam_peak / C)
    print(f"{temperature:10.0f} {lam_peak * 1e9:17.1f} {nu_peak / 1e12:14.2f} "
          f"{C / nu_peak * 1e9:17.1f} {products[-1]:22.4f}")

sun = 5772.0
lam = np.linspace(100e-9, 3000e-9, 1000)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
nu = C / lam
left.plot(nu / 1e12, radiation.planck_u_nu(nu, sun), color="#2563eb")
left.axvline(radiation.wien_peak_nu(sun) / 1e12, color="#2563eb", label="own peak")
left.axvline(C / radiation.wien_peak_lambda(sun) / 1e12, color="#dc2626", ls="--",
             label="the other panel's peak")
left.set_xlabel("frequency (THz)")
left.set_ylabel("u_nu (J s m^-3)")
left.legend()
right.plot(lam * 1e9, radiation.planck_u_lambda(lam, sun), color="#dc2626")
right.axvline(radiation.wien_peak_lambda(sun) * 1e9, color="#dc2626", label="own peak")
right.axvline(C / radiation.wien_peak_nu(sun) * 1e9, color="#2563eb", ls="--",
              label="the other panel's peak")
right.set_xlabel("wavelength (nm)")
right.set_ylabel("u_lambda (J m^-4)")
right.legend()
for ax in (left, right):
    ax.set_title("5772 K")
plt.tight_layout()
plt.show()

The ratio is $0.568$ at every temperature: the photon at the frequency peak has a wavelength
$1.76$ times $\lambda_{\max}$. At the Sun's temperature the wavelength peak is green and the
frequency peak is infrared. Both graphs are right. "The peak" is a property of the graph, because
a spectral density is a density.

## Part 6 — The Stefan–Boltzmann law by quadrature

Integrate $u_\nu$ numerically, with no reference to $\pi^4/15$, and compare with $aT^4$. Then
halve the grid spacing repeatedly and watch the error; then integrate $u_\lambda$ instead, and
fit the exponent of $u(T)$.

In [ ]:
exact = radiation.energy_density(300.0)
previous = None
print("intervals   relative error   ratio to previous")
for n in (64, 128, 256, 512, 1024, 2048, 4096):
    error = relative_error(radiation.integrate_u_nu(300.0, n), exact)
    ratio = f"{previous / error:8.1f}" if previous else ""
    print(f"{n:9d} {error:16.2e}   {ratio}")
    previous = error

for temperature in (2.725, 300.0, 5772.0):
    by_nu = radiation.integrate_u_nu(temperature)
    by_lambda = radiation.integrate_u_lambda(temperature)
    print(f"T = {temperature:7.3f} K: per frequency {by_nu:.6e}, per wavelength {by_lambda:.6e}, "
          f"a T^4 {radiation.energy_density(temperature):.6e} J/m^3")

temps = np.geomspace(100.0, 3000.0, 9)
exponent = scaling_exponent(temps, [radiation.integrate_u_nu(t) for t in temps])
print(f"log-log slope of u(T): {exponent:.12f};  300 K -> 600 K multiplies u by "
      f"{radiation.integrate_u_nu(600.0) / radiation.integrate_u_nu(300.0):.4f}")
gas = radiation.photon_gas(300.0)
print(f"photon gas at 300 K: n = {gas.number_density:.3e} m^-3, P = {gas.pressure:.3e} Pa, "
      f"energy per photon = {gas.energy_per_photon / (K_B * 300):.4f} k_B T")

The error falls sixteen-fold per halving — Simpson's rule's fourth order — and the integral
reaches $aT^4$, confirming the $\pi^4/15$ the closed form quoted. The wavelength integral, on an
unrelated grid, gives the same energy. The exponent is $4$.

## Part 7 — Silver at low temperature

The data file holds silver's heat capacity at constant pressure from $1$ to $300\ \mathrm{K}$,
tabulated by NIST (Smith and Fickett, 1995), per kilogram. Converted to a mole, below about
$6\ \mathrm{K}$ it should be the sum of the conduction electrons' $\gamma T$ and Debye's
$\beta T^3$ — so $C/T$ plotted against $T^2$ should be a straight line.

**Your prediction:** what slope will the raw data have on a log–log plot between $1$ and
$3\ \mathrm{K}$?

*(write here before running the next cell)*

In [ ]:
MOLAR_MASS = 0.1078682  # kg/mol
t_ag, cp_per_kg = np.genfromtxt(DATA / "16-silver-heat-capacity.csv", delimiter=",",
                                comments="#").T
c_ag = cp_per_kg * MOLAR_MASS  # J/(mol K)

low = t_ag <= 6.0
(beta, gamma), cov = np.polyfit(t_ag[low] ** 2, c_ag[low] / t_ag[low], 1, cov=True)
theta_0 = (12 * np.pi**4 * R_GAS / (5 * beta)) ** (1 / 3)
theta_0_err = theta_0 * np.sqrt(cov[0, 0]) / (3 * beta)
print(f"gamma = {gamma * 1e3:.4f} +- {np.sqrt(cov[1, 1]) * 1e3:.4f} mJ/(mol K^2)   (paper: 0.6409)")
print(f"beta  = {beta * 1e3:.5f} mJ/(mol K^4)  ->  "
      f"theta_D = {theta_0:.1f} +- {theta_0_err:.1f} K   (paper: 226.6)")

lattice = c_ag - gamma * t_ag
raw_window, lattice_window = (t_ag >= 1) & (t_ag <= 3), (t_ag >= 1) & (t_ag <= 5)
slope_raw = np.polyfit(np.log(t_ag[raw_window]), np.log(c_ag[raw_window]), 1)[0]
slope_lattice = np.polyfit(np.log(t_ag[lattice_window]), np.log(lattice[lattice_window]), 1)[0]
print(f"log-log slope, raw data, 1-3 K: {slope_raw:.2f};  "
      f"with gamma T removed, 1-5 K: {slope_lattice:.2f}")
print(f"electrons' share of C at 1 K: {gamma * 1 / c_ag[0]:.0%}")

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(t_ag[low] ** 2, c_ag[low] / t_ag[low] * 1e3, "o", color="k")
grid = np.linspace(0, 36, 50)
left.plot(grid, (gamma + beta * grid) * 1e3, color="#2563eb")
left.set_xlabel("T^2 (K^2)")
left.set_ylabel("C / T (mJ mol^-1 K^-2)")
right.loglog(t_ag, c_ag, "o", ms=3, color="k", label="C_P, measured")
right.loglog(t_ag, lattice, "o", ms=3, mfc="none", color="#2563eb", label="minus gamma T")
right.axhline(3 * R_GAS, color="grey", ls="--", label="3R")
right.set_xlabel("T (K)")
right.set_ylabel("C (J mol^-1 K^-1)")
right.legend()
plt.tight_layout()
plt.show()

A straight line, as promised: its intercept is the electrons, its slope the lattice. Below about
$2\ \mathrm{K}$ the electrons carry most of the heat capacity, and a large share up to
$4\ \mathrm{K}$, which is why the raw slope is far from $3$ — a metal's failure mode, in real
numbers. With $\gamma T$ removed the slope is $3$.

## Part 8 — Einstein against Debye

Fit each model's one parameter to the lattice heat capacity between $4$ and $100\ \mathrm{K}$,
on fractional residuals, and look at the residuals. Then ask the Debye formula, point by point,
which $\theta_D$ each measurement needs.

**Your prediction:** which model will fail, and where?

*(write here before running the next cell)*

In [ ]:
window = (t_ag >= 4) & (t_ag <= 100)
debye = radiation.fit_debye_temperature(t_ag[window], lattice[window], N_A)
einstein = radiation.fit_einstein_temperature(t_ag[window], lattice[window], N_A)
for name, fit in (("Debye:   ", debye), ("Einstein:", einstein)):
    print(f"{name} theta = {fit.value:6.1f} +- {fit.sigma:.1f} K, "
          f"rms residual {fit.rms_residual:.1%}")


def running_theta(temperature, heat_capacity):
    # The theta_D for which the Debye formula reproduces one measured point (bisection in log).
    low_theta, high_theta = 10.0, 3000.0
    for _ in range(60):
        mid = np.sqrt(low_theta * high_theta)
        if radiation.debye_heat_capacity(temperature, mid, N_A) > heat_capacity:
            low_theta = mid
        else:
            high_theta = mid
    return mid


running = {t: running_theta(t, c) for t, c in zip(t_ag, lattice, strict=True) if 2 <= t <= 120}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
grid = np.geomspace(1, 300, 300)
axes[0].loglog(t_ag, lattice, "o", ms=3, color="k", label="silver, lattice")
axes[0].loglog(grid, radiation.debye_heat_capacity(grid, debye.value, N_A), color="#2563eb",
               label=f"Debye {debye.value:.0f} K")
axes[0].loglog(grid, radiation.einstein_heat_capacity(grid, einstein.value, N_A), color="#dc2626",
               label=f"Einstein {einstein.value:.0f} K")
axes[0].set_ylim(1e-4, 40)
axes[0].set_xlabel("T (K)")
axes[0].set_ylabel("C (J mol^-1 K^-1)")
axes[0].legend()
axes[1].semilogx(t_ag[window], debye.residuals, "o-", color="#2563eb", label="Debye")
axes[1].semilogx(t_ag[window], einstein.residuals, "o-", color="#dc2626", label="Einstein")
axes[1].axhline(0, color="grey")
axes[1].set_xlabel("T (K)")
axes[1].set_ylabel("model / data - 1")
axes[1].legend()
axes[2].plot(list(running), list(running.values()), "o-", color="#2563eb")
axes[2].set_xlabel("T (K)")
axes[2].set_ylabel("running theta_D (K)")
plt.tight_layout()
plt.show()

Einstein's curve collapses exponentially below about $20\ \mathrm{K}$, where silver's does not:
one frequency cannot be cheap and dear at once. Debye follows the data within about $10\%$ — but
the running $\theta_D$ wanders between about $210$ and $226\ \mathrm{K}$. A one-parameter
caricature of the phonon spectrum survives the test only if its parameter is allowed to drift.

## Part 9 — The cosmic microwave background

The second data file is the spectrum of the cosmic microwave background measured by the FIRAS
instrument on COBE, from NASA's LAMBDA archive. Fit the temperature of a Planck curve, in
radiance $B_\nu = (c/4\pi)\,u_\nu$, and look at what is left over.

**Your prediction:** the data are plotted per unit frequency, and peak near
$5.4\ \mathrm{cm^{-1}}$, a wavelength of about $1.85\ \mathrm{mm}$. Where will the same spectrum,
expressed per unit *wavelength*, peak?

*(write here before running the next cell)*

In [ ]:
wavenumber, intensity, residual, sigma_k, _ = np.genfromtxt(
    DATA / "16-cmb-firas-monopole.csv", delimiter=",", comments="#").T
nu_cmb = 100 * C * wavenumber                 # Hz
radiance = intensity * 1e-20                  # W m^-2 Hz^-1 sr^-1
sigma = sigma_k * 1e-3 * 1e-20

cmb = radiation.fit_blackbody_temperature(nu_cmb, radiance, sigma)
print(f"T = {cmb.value:.6f} +- {cmb.sigma:.6f} K (statistical);  chi^2 = {cmb.chi_squared:.1f} "
      f"for {cmb.degrees_of_freedom} degrees of freedom")
per_lambda = radiance * nu_cmb**2 / C          # B_lambda = B_nu |d nu / d lambda|
print(f"data peak per unit frequency: {wavenumber[np.argmax(radiance)]:.2f} cm^-1 "
      f"({10 / wavenumber[np.argmax(radiance)]:.2f} mm);  per unit wavelength: "
      f"{10 / wavenumber[np.argmax(per_lambda)]:.2f} mm")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                                  gridspec_kw={"height_ratios": [2, 1]})
fine = np.linspace(nu_cmb.min(), nu_cmb.max(), 400)
top.plot(fine / (100 * C), radiation.planck_radiance_nu(fine, cmb.value) * 1e20, color="#2563eb",
         label=f"Planck, {cmb.value:.4f} K")
top.errorbar(wavenumber, intensity, yerr=sigma_k * 1e-3, fmt="o", ms=3, color="k", label="FIRAS")
top.set_ylabel("intensity (MJy / sr)")
top.legend()
bottom.errorbar(wavenumber, cmb.residuals, yerr=1.0, fmt="o", ms=3, color="k")
bottom.axhline(0, color="#2563eb")
bottom.set_xlabel("wavenumber (cm^-1)")
bottom.set_ylabel("residual / sigma")
plt.tight_layout()
plt.show()

A Planck curve fits the 43 points with $\chi^2$ close to the number of degrees of freedom: the
shape is right, to a part in ten thousand of the peak. The statistical error on $T$ is tiny, and
it is not the whole story. The archive built this spectrum as a $2.725\ \mathrm{K}$ Planck curve
plus the measured residuals, so the fit recovers the temperature the residuals were taken
against; the absolute temperature is limited by FIRAS's calibration, and the published value is
$2.72548 \pm 0.00057\ \mathrm{K}$, and the $\chi^2$ treats the published errors as independent,
which they are not quite. And the peak, again, depends on the axis: the measured points peak at
$5.45\ \mathrm{cm^{-1}}$, a wavelength of $1.83\ \mathrm{mm}$, per unit frequency, and at
$1.05\ \mathrm{mm}$ per unit wavelength (the fitted curve: $1.87$ and $1.06\ \mathrm{mm}$; $43$
frequencies are a coarse grid to read a peak from).

### Measurement

Quote two results, each with its uncertainty and the assumptions it rests on:

- $\theta_D$(silver) from Part 7's low-temperature line, against $226.6\ \mathrm{K}$ (Smith and
  Fickett) — and say how much it moves if the window ends at $8\ \mathrm{K}$ instead of $6$.
- $T_{\mathrm{CMB}}$ from Part 9, against $2.725\ \mathrm{K}$ — and say which error dominates.

## Part 10 — The automated checks

In [ ]:
# 1. The occupation is module 12's sum.
levels = 4e-21 * np.arange(4000)
assert relative_error(radiation.mode_occupation(4e-21, 300.0),
                      partition.level_sums(levels, 300.0).energy / 4e-21) < 1e-10

# 2. Counting modes approaches the smooth formula, from below.
assert 0 < shortfalls[320] < 0.01 and shortfalls[320] < shortfalls[40] / 4

# 3. Two peaks, one ratio.
assert all(abs(p - radiation.DUAL_PEAK_RATIO) < 1e-3 for p in products)

# 4. T^4, and the same energy in either variable.
assert abs(exponent - 4.0) < 1e-6
assert relative_error(radiation.integrate_u_lambda(300.0), radiation.integrate_u_nu(300.0)) < 1e-8

# 5. Silver: the electrons and the lattice, from the low-temperature line.
assert 0.6 < gamma * 1e3 < 0.7
assert 220 < theta_0 < 232
assert 2.8 < slope_lattice < 3.2 and slope_raw < 2.2

# 6. The data prefer Debye, by far.
assert debye.rms_residual < 0.15 < 0.3 < einstein.rms_residual
assert 200 < debye.value < 235

# 7. The microwave background is a blackbody near 2.725 K.
assert abs(cmb.value - 2.725) < 0.001
assert cmb.chi_squared / cmb.degrees_of_freedom < 2.0
print("all checks passed")

## Part 11 — Explore it yourself

Two explorers; set the controls and press **Run Interact**.

**The blackbody explorer.** The spectrum at the temperature you choose, per unit frequency and
per unit wavelength side by side, each with its own peak (solid) and the other panel's peak
carried over (dashed). Things to try: $300\ \mathrm{K}$ (a room), $2800\ \mathrm{K}$ (a lamp
filament), $5772\ \mathrm{K}$ (the Sun), $10\,000\ \mathrm{K}$ (a hot star); the classical curve
at each; logarithmic axes, which show the whole spectrum at once.

In [ ]:
import ipywidgets as widgets


def blackbody(temperature=5772.0, classical=False, log_axes=False):
    nu_peak, lam_peak = radiation.wien_peak_nu(temperature), radiation.wien_peak_lambda(temperature)
    nu = np.linspace(0.01, 6.0, 1200) * nu_peak
    lam = np.linspace(0.05, 8.0, 1200) * lam_peak
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
    left.plot(nu, radiation.planck_u_nu(nu, temperature), color="#2563eb", lw=2)
    right.plot(lam * 1e9, radiation.planck_u_lambda(lam, temperature), color="#dc2626", lw=2)
    if classical:
        left.plot(nu, radiation.rayleigh_jeans_u_nu(nu, temperature), "--", color="grey")
        right.plot(lam * 1e9, radiation.rayleigh_jeans_u_nu(C / lam, temperature) * C / lam**2,
                   "--", color="grey")
    left.axvline(nu_peak, color="#2563eb")
    left.axvline(C / lam_peak, color="#dc2626", ls="--")
    right.axvline(lam_peak * 1e9, color="#dc2626")
    right.axvline(C / nu_peak * 1e9, color="#2563eb", ls="--")
    for ax in (left, right):
        if log_axes:
            ax.set_xscale("log")
            ax.set_yscale("log")
    if not log_axes:
        left.set_ylim(0, 1.3 * radiation.planck_u_nu(nu_peak, temperature))
        right.set_ylim(0, 1.3 * radiation.planck_u_lambda(lam_peak, temperature))
    left.set_xlabel("frequency (Hz)")
    left.set_ylabel("u_nu (J s m^-3)")
    right.set_xlabel("wavelength (nm)")
    right.set_ylabel("u_lambda (J m^-4)")
    plt.tight_layout()
    plt.show()
    visible = radiation.band_fraction(temperature, 380e-9, 750e-9)
    print(f"T = {temperature:.0f} K: lambda_max = {lam_peak * 1e9:.0f} nm; "
          f"nu_max = {nu_peak / 1e12:.1f} THz (the frequency of {C / nu_peak * 1e9:.0f} nm light);")
    print(f"u = {radiation.energy_density(temperature):.3e} J/m^3; visible fraction {visible:.2%}")


widgets.interact_manual(
    blackbody,
    temperature=widgets.FloatLogSlider(value=5772.0, base=10, min=2, max=4, step=0.01),
    classical=widgets.Checkbox(value=False, description="classical curve"),
    log_axes=widgets.Checkbox(value=False, description="log axes"),
)

**The heat-capacity explorer.** Silver's lattice heat capacity against the Einstein and Debye
curves for the temperatures you choose, with the fractional residuals beneath. Things to try:
make Einstein fit near $100\ \mathrm{K}$, then switch to log–log and look at $5\ \mathrm{K}$; find
the $\theta_D$ that is best below $10\ \mathrm{K}$, then the one best near $25\ \mathrm{K}$.

In [ ]:
def solids(theta_e=160.0, theta_d=220.0, log_log=True):
    grid = np.geomspace(1.0, 300.0, 300)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                                      gridspec_kw={"height_ratios": [2, 1]})
    top.plot(t_ag, lattice / (3 * R_GAS), "o", ms=3, color="k", label="silver (lattice)")
    top.plot(grid, radiation.einstein_heat_capacity(grid, theta_e, N_A) / (3 * R_GAS),
             color="#dc2626", label=f"Einstein, {theta_e:.0f} K")
    top.plot(grid, radiation.debye_heat_capacity(grid, theta_d, N_A) / (3 * R_GAS),
             color="#2563eb", label=f"Debye, {theta_d:.0f} K")
    top.set_ylabel("C / 3R")
    top.legend()
    bottom.plot(t_ag, radiation.einstein_heat_capacity(t_ag, theta_e, N_A) / lattice - 1, "o-",
                ms=3, color="#dc2626")
    bottom.plot(t_ag, radiation.debye_heat_capacity(t_ag, theta_d, N_A) / lattice - 1, "o-",
                ms=3, color="#2563eb")
    bottom.axhline(0, color="grey")
    bottom.set_ylim(-1.05, 1.0)
    bottom.set_ylabel("model / data - 1")
    bottom.set_xlabel("T (K)")
    if log_log:
        top.set_xscale("log")
        top.set_yscale("log")
        top.set_ylim(1e-6, 2)
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    solids,
    theta_e=widgets.FloatSlider(value=160.0, min=50.0, max=400.0, step=1.0),
    theta_d=widgets.FloatSlider(value=220.0, min=50.0, max=400.0, step=1.0),
    log_log=widgets.Checkbox(value=True, description="log-log"),
)

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "16-radiation-solids.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. In Part 5, both graphs showed the same radiation. Explain in your own words why they peak at
   different photons, and what *is* the same in both.
3. Your measurement of $\theta_D$ quotes an uncertainty. List every assumption it rests on that
   the uncertainty does *not* include.
4. Silver's heat capacity at $1\ \mathrm{K}$ is mostly electronic. Why does that not contradict
   the claim that the lattice's heat capacity falls as $T^3$?

**Your answers:**

1.
2.
3.
4.